# Gemini + RAG demo - reproducibility analysis of a single PDF

Uses `gemini_rag.GeminiPaperAnalyst` to extract a structured `PaperProfile` from any PDF on disk.

**Setup**

```bash
uv pip install google-genai pydantic
export GEMINI_API_KEY=your-key-here   # or GOOGLE_API_KEY
```

The pipeline runs five independent, schema-constrained queries against the PDF (methodology, datasets, artefacts, figures/tables, parameters) at `temperature=0` with a system instruction that pins answers to the attached document. Each extracted item carries a `source_quote` field so you can audit for hallucination.

## 0. Available paired papers

Lists every row of `data/rescience_bibtex_table.xlsx` that has a PDF in **both** the `RESCIENCE C` and `ORIGINAL` sections. Pick one of these filenames as the `PDF_PATH` in section 2.

In [1]:
import pandas as pd
from pathlib import Path

XLSX = Path("data/rescience_bibtex_table.xlsx")
RESCIENCE_CACHE = Path("data/pdf_cache")
ORIGINAL_CACHE = Path("data/pdf_original_cache")

# The xlsx has a two-row header: (section, field). Flatten to prefixed names so we
# can address every column unambiguously -- e.g. "rescience_filename", "original_filename".
df_all = pd.read_excel(XLSX, header=[0, 1])
def _flatten(col: tuple[str, str]) -> str:
    section, field = col
    if section.startswith("Unnamed") or field.startswith("Unnamed"):
        return "idx"
    prefix = {"RESCIENCE C": "rescience", "ORIGINAL": "original"}.get(section, section.lower())
    return f"{prefix}_{field}"
df_all.columns = [_flatten(c) for c in df_all.columns]

# Use the filename columns (authoritative) rather than the pdf_file download-marker columns.
def _resolve(cache: Path, name) -> Path | None:
    if not isinstance(name, str) or not name.strip() or name.strip().lower() == "nan":
        return None
    p = cache / name.strip()
    return p if p.is_file() else None

df_all["_rescience_path"] = df_all["rescience_filename"].map(lambda n: _resolve(RESCIENCE_CACHE, n))
df_all["_original_path"]  = df_all["original_filename"].map(lambda n: _resolve(ORIGINAL_CACHE, n))

paired = df_all[df_all["_rescience_path"].notna() & df_all["_original_path"].notna()].copy()
missing_rescience = df_all["_rescience_path"].isna().sum()
missing_original  = df_all["_original_path"].isna().sum()

paired_view = pd.DataFrame({
    "rescience_pdf":   paired["rescience_filename"].values,
    "original_pdf":    paired["original_filename"].values,
    "rescience_title": paired["rescience_title"].values,
    "original_title":  paired["original_title"].values,
})
print(f"Total rows in xlsx:            {len(df_all)}")
print(f"Rescience PDFs missing on disk: {missing_rescience}")
print(f"Original PDFs missing on disk:  {missing_original}")
print(f"Paired (both PDFs available):   {len(paired_view)}\n")
paired_view

Total rows in xlsx:            222
Rescience PDFs missing on disk: 0
Original PDFs missing on disk:  7
Paired (both PDFs available):   215



,rescience_pdf,original_pdf,rescience_title,original_title
0,2026_01_article.pdf,2026_01_article.pdf,Learning with Noisy Labels [Re]visited,Learning with Noisy Labels Revisited: A Study ...
1,2026_02_article.pdf,2026_02_article.pdf,[Re] Curve-Fitting with Piecewise Parametric C...,Curve-Fitting with Piecewise Parametric Cubics
2,2025_01_article.pdf,2025_01_article.pdf,[Re] Model of thalamocortical slow-wave sleep ...,Model of thalamocortical slow-wave sleep oscil...
3,2025_02_article.pdf,2025_02_article.pdf,[Re] Learning Fair Graph Representations via A...,Learning Fair Graph Representations via Automa...
4,2025_03_article.pdf,2025_03_article.pdf,[Re] Network Deconvolution,Network Deconvolution
...,...,...,...,...
210,2016_04_article.pdf,2016_04_article.pdf,[Re] Multiple dynamical modes of thalamic rela...,Multiple dynamical modes of thalamic relay neu...
211,2016_05_article.pdf,2016_05_article.pdf,[Re] Chaos in a long-term experiment with a pl...,Chaos in a long-term experiment with a plankto...
212,2016_06_article.pdf,2016_06_article.pdf,[Re] Least-cost modelling on irregular landsca...,Least-cost modelling on irregular landscape gr...
213,2016_07_article.pdf,2016_07_article.pdf,[Re] Speed/accuracy trade-off between the habi...,Speed/accuracy trade-off between the habitual ...


## 1. Imports and setup

In [2]:
import json
from pathlib import Path

from gemini_rag import GeminiPaperAnalyst, PaperProfile

## 2. Point at a specific PDF

Change `PDF_PATH` to any paper you want to analyse. Defaults to one of the cached originals.

In [3]:
# Pick a row from the paired_view above, then resolve to a cached PDF.
# Swap ORIGINAL_CACHE <-> RESCIENCE_CACHE to analyse the replication instead of the original.
PDF_NAME = paired_view.iloc[-1]["original_pdf"]   # e.g. "2023_04_article.pdf"
PDF_PATH = ORIGINAL_CACHE / PDF_NAME
assert PDF_PATH.exists(), f"missing: {PDF_PATH}"
print(f"PDF: {PDF_PATH}  ({PDF_PATH.stat().st_size / 1024:.1f} KB)")

PDF: data/pdf_original_cache/2015_01_article.pdf  (1758.2 KB)


## 3. Run the analyst

In [4]:
analyst = GeminiPaperAnalyst(model="gemini-3-flash-preview")
profile: PaperProfile = analyst.analyze(PDF_PATH)

Uploading 2015_01_article.pdf to Gemini Files API...
  querying: header ...
  querying: methodology ...
  querying: datasets ...
  querying: artefacts ...
  querying: figures_tables ...
  querying: parameters ...


## 4. Inspect the PaperProfile

In [5]:
print(f"Title:   {profile.title}")
print(f"Authors: {', '.join(profile.authors)}")
print(f"Methodology steps: {len(profile.methodology_steps)}")
print(f"Datasets:          {len(profile.datasets)}")
print(f"Figures:           {len(profile.figures_to_reproduce)}")
print(f"Tables:            {len(profile.tables_to_reproduce)}")
print(f"Hyperparameters:   {len(profile.hyperparameters)}")
print(f"Repository links:  {profile.repository_links}")

Title:   Interaction between cognitive and motor cortico-basal ganglia loops during decision making: a computational study
Authors: M. Guthrie, A. Leblois, A. Garenne, T. Boraud
Methodology steps: 7
Datasets:          1
Figures:           8
Tables:            1
Hyperparameters:   18
Repository links:  []


### Methodology steps (with grounding quotes)

In [6]:
for step in profile.methodology_steps:
    print(f"[{step.order}] {step.description}")
    if step.tools_mentioned:
        print(f"    tools: {', '.join(step.tools_mentioned)}")
    if step.source_quote:
        print(f'    quote: "{step.source_quote}"')
    print()

[1] Define the center-out motor task parameters, including 120 trials and four cues with reward probabilities {P(R)[0, 0.33, 0.66, 1]}.
    quote: "During a simulation, consisting of 120 trials, four cues were used, each with a different reward probability {P(R)[0, 0.33, 0.66, 1]}"

[2] Initialize the two-level cortico-basal ganglia loop model, setting synaptic weights to 0.5 and defining neuronal parameters for each structure.
    tools: Delphi 7 (Borland 2001)
    quote: "At the start of each run, all synaptic weights were initialized to 0.5 (SD 0.005)."

[3] Run the model for a 500 ms settling period at the start of each trial using a first-order Euler algorithm with a 1-ms step.
    tools: Delphi 7 (Borland 2001)
    quote: "Each trial starts with a period of 500 ms to allow the network to settle, after which the 2 cues are presented."

[4] Present two cues and perform action selection based on cortical ensemble activation reaching a threshold of 40 sp/s greater than others.
    to

### Datasets

In [10]:
for ds in profile.datasets:
    print(f"- {ds.name}  ({ds.availability.value})")
    if ds.url:
        print(f"    url: {ds.url}")
    if ds.source_quote:
        print(f'    quote: "{ds.source_quote}"')

- Pasquereau et al. (2007) electrophysiological data  (not_mentioned)
    quote: "Recent electrophysiological data from Pasquereau et al. (2007), in primates, suggest that, in a two-armed bandit task, two separable processes occur during action selection."


## 5. Save the profile to JSON

In [8]:
import re

stem = PDF_PATH.stem  # e.g. "2026_01_article"
existing = list(PDF_PATH.parent.glob(f"{stem}_exe*.profile.json"))
used = [
    int(m.group(1))
    for p in existing
    if (m := re.match(rf"{re.escape(stem)}_exe(\d+)\.profile\.json$", p.name))
]
exe_n = (max(used) + 1) if used else 1

out = PDF_PATH.parent / f"{stem}_exe{exe_n}.profile.json"
out.write_text(profile.model_dump_json(indent=2))
print(f"wrote: {out}")

wrote: data/pdf_original_cache/2015_01_article_exe1.profile.json


## 6. Token usage and cost estimate

`analyst.usage_log` captures `usage_metadata` from every call since the last `analyze()`. `analyst.usage_summary()` aggregates it and applies the paid-tier rates in `gemini_rag.GEMINI_PRICING_USD_PER_MTOK`.

Rates drift - always cross-check against https://ai.google.dev/pricing and your AI Studio billing dashboard for authoritative numbers.

In [9]:
summary = analyst.usage_summary()

print(f"Model: {summary['model']}")
print(f"API calls: {summary['calls']}")
print()
print(f"{'query':<18} {'input':>10} {'output':>10}")
print("-" * 40)
for q in summary["per_query"]:
    print(f"{q['name']:<18} {q['input_tokens']:>10,} {q['output_tokens']:>10,}")
print("-" * 40)
print(f"{'TOTAL':<18} {summary['input_tokens']:>10,} {summary['output_tokens']:>10,}")
print(f"Grand total tokens: {summary['total_tokens']:,}")
print()
if summary["total_usd"] is None:
    print(f"No price table entry for model {summary['model']!r} - add one to GEMINI_PRICING_USD_PER_MTOK.")
else:
    print(
        f"Estimated cost (USD):  input ${summary['input_usd']:.6f}  "
        f"+  output ${summary['output_usd']:.6f}  "
        f"=  ${summary['total_usd']:.6f}"
    )

Model: gemini-3-flash-preview
API calls: 6

query                   input     output
----------------------------------------
header                  8,427         70
methodology             8,736        701
datasets                8,463        173
artefacts               8,450         28
figures_tables          8,446      1,424
parameters              8,446      1,131
----------------------------------------
TOTAL                  50,968      3,527
Grand total tokens: 54,495

No price table entry for model 'gemini-3-flash-preview' - add one to GEMINI_PRICING_USD_PER_MTOK.
